# Agentic Systems — recorded execution outputs

This executable evidence viewer displays saved outputs from two actual ADA bundles (SDK 2.1.8). It reads committed JSON and performs no inference. It is not the original exported execution notebook. The complete application source is [ADA_Time_Series_Discovery_219_Core.ipynb](ADA_Time_Series_Discovery_219_Core.ipynb).

The supplied Native PDF shows a separate failed rerun (page 25); the OpenAI Agents PDF shows a successful run (page 21). Native's earlier PASS comes from its original JSON bundle. Both are retained without relabeling the failed run.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
native = json.loads(Path("outputs/native.json").read_text())
openai = json.loads(Path("outputs/openai_agents.json").read_text())
profiles = {"Native": native, "OpenAI Agents": openai}


## 1 · Tool and graph-backed Agent

In [2]:
display(native['results']['01_comparison']['data']['measurement'])

{'kind': 'compare',
 'value': 7.389999999999986,
 'details': {'earlier_mean': 84.82000000000001,
  'later_mean': 92.21,
  'window_size': 5}}

## 2 · Skill and model-backed Agent

In [3]:
for name, profile in profiles.items():
    key = profile['profile'] + '_description'
    print(name, profile['results'][key]['status'])
    display([e['output'] for e in profile['results'][key]['tool_events']])

Native completed
OpenAI Agents completed


[{'kind': 'object',
  'tool_name': 'profile_series',
  'ok': True,
  'data': {'profile': {'count': 10,
    'average': 88.515,
    'minimum': 80.09,
    'maximum': 97.84}},
  'meta': {'tool_name': 'profile_series',
   'serializer': 'BedrockRuntime.ToolEnvelope.v1',
   'timestamp_utc': '2026-11-11T23:52:11.134632+00:00',
   'validated_input': {'values': [80.09,
     83.88,
     85.97,
     88.18,
     85.98,
     88.71,
     87.69,
     92.28,
     94.53,
     97.84]}}}]

[{'data': {'profile': {'count': 10,
    'average': 88.515,
    'minimum': 80.09,
    'maximum': 97.84}}}]

## 3 · System and validated state

In [4]:
r = native['results']['03_discovery']
display({k:r['data'][k] for k in ['selected_tool','measurement','analysis_checked']})

{'selected_tool': 'compare_windows',
 'measurement': {'kind': 'compare',
  'value': 7.389999999999986,
  'details': {'earlier_mean': 84.82000000000001,
   'later_mean': 92.21,
   'window_size': 5}},
 'analysis_checked': True}

## 4 · System composition and report

In [5]:
print(native['results']['04_reviewed']['data']['report'])

2005–2014: difference=+7.390 index points; review=no_candidate; threshold=10.0. Descriptive screening only; no statistical significance or causal claim.


## 5 · Environment and Memory

In [6]:
for name, profile in profiles.items():
    r = profile['results'][profile['profile'] + '_environment']
    print(name, r['status'])
    display(pd.DataFrame([{'period': f"{x['years'][0]}–{x['years'][-1]}",
        'difference':x['measurement']['value'], 'verdict':x['review']['verdict'],
        'prior_episodes':x['review']['previous_episodes']} for x in r['data']['outputs']]))
    display(r['data']['summary'])

Native completed
OpenAI Agents completed


,period,difference,verdict,prior_episodes
0,1961–1970,5.928,no_candidate,0
1,1971–1980,7.850,no_candidate,1
2,1981–1990,3.142,no_candidate,2
3,1991–2000,10.034,candidate,3


{'state': {'processed': 4, 'candidate_found': True},
 'processed': 4,
 'terminated': True,
 'truncated': False,
 'reason': 'stop'}

,period,difference,verdict,prior_episodes
0,1961–1970,5.928,no_candidate,0
1,1971–1980,7.850,no_candidate,1
2,1981–1990,3.142,no_candidate,2
3,1991–2000,10.034,candidate,3


{'state': {'processed': 4, 'candidate_found': True},
 'processed': 4,
 'terminated': True,
 'truncated': False,
 'reason': 'stop'}

## 6 · Eval: result and trajectory

In [7]:
display(pd.DataFrame(native['summary']['local_evaluations']))
for name, profile in profiles.items():
    r = profile['results'][profile['profile'] + '_eval']
    print(name, r['status'])
    display(r['data'])

Native completed
OpenAI Agents completed


,case,expected_acceptance,actual,test
0,agent,True,True,PASS
1,system,True,True,PASS
2,environment,True,True,PASS
3,wrong expected answer,False,False,PASS
4,odd windows rejected,True,True,PASS


{'total': 1,
 'passed': 1,
 'failed': 0,
 'cases': [{'name': '0',
   'ok': True,
   'input': {'question': 'Compare the earlier and later observations and show supporting measurements.',
    'years': [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014],
    'values': [80.09,
     83.88,
     85.97,
     88.18,
     85.98,
     88.71,
     87.69,
     92.28,
     94.53,
     97.84]},
   'execution_id': 'run-47a3bd985e824855a233eb09977bf251',
   'checks': [{'ok': True, 'path': 'ok'}, {'ok': True}, {'ok': True}],
   'judge': None}]}

{'total': 1,
 'passed': 1,
 'failed': 0,
 'cases': [{'name': '0',
   'ok': True,
   'input': {'question': 'Compare the earlier and later observations and show supporting measurements.',
    'years': [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014],
    'values': [80.09,
     83.88,
     85.97,
     88.18,
     85.98,
     88.71,
     87.69,
     92.28,
     94.53,
     97.84]},
   'execution_id': 'run-eb07c33e785044589b2fa5b33f1d6b57',
   'checks': [{'ok': True, 'path': 'ok'}, {'ok': True}, {'ok': True}],
   'judge': None}]}

## 7 · Framework portability and run history

In [8]:
display(pd.DataFrame([{'framework': name, **profile['summary']['live'][0]}
    for name, profile in profiles.items()]))
failed_rerun = json.loads(Path("outputs/native_failed_rerun.json").read_text())
print("Separate Native rerun, observed in supplied PDF:")
display(pd.DataFrame(failed_rerun['results']))
print(failed_rerun['interpretation'])

Separate Native rerun, observed in supplied PDF:
A different failed rerun; this PDF does not establish the root cause. The earlier successful Native run is retained separately in native.json.


,framework,profile,description_selection,environment,environment_check,discovery_eval,http_attempts_reserved,status,http_attempts_this_profile
0,Native,bedrock_native,PASS,completed,True,completed,30,NaN,NaN
1,OpenAI Agents,bedrock_openai_agents,PASS,completed,True,completed,80,PASS,20.0


,name,status,invariants
0,bedrock_native_description,failed,True
1,bedrock_native_environment,failed,True
2,bedrock_native_eval,failed,True
